In [1]:
import os
import sys
sys.path.append(os.getcwd())

import hydra
from hydra import compose, initialize
from omegaconf import DictConfig
import torch
import nibabel as nib
import numpy as np
from matplotlib import pyplot as plt

from dataset.dataloader import get_loader



In [2]:

initialize(version_base=None, config_path="config")

# Load config
cfg = compose(config_name="base_cfg")

In [3]:
train_loader, _, _ = get_loader(cfg.dataset)

val len 1


/home/rpinise1/.conda/envs/synth-env/lib/python3.10/site-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [4]:
d = next(iter(train_loader))

In [11]:
import numpy as np

def paste_tumor(
    full_mask,
    tumor_patch,
    centroid,
):
    """
    full_mask: full-size target volume
    tumor_patch: generated tumor volume
    centroid: (x, y, z) in full-volume coordinates
    """

    output = full_mask.copy()

    cx, cy, cz = centroid

    tx, ty, tz = tumor_patch.shape

    # Compute insertion bounds
    x0 = cx - tx // 2
    y0 = cy - ty // 2
    z0 = cz - tz // 2

    x1 = x0 + tx
    y1 = y0 + ty
    z1 = z0 + tz

    # Clip to valid region
    fx0 = max(x0, 0)
    fy0 = max(y0, 0)
    fz0 = max(z0, 0)

    fx1 = min(x1, output.shape[0])
    fy1 = min(y1, output.shape[1])
    fz1 = min(z1, output.shape[2])

    # Corresponding patch region
    px0 = fx0 - x0
    py0 = fy0 - y0
    pz0 = fz0 - z0

    px1 = px0 + (fx1 - fx0)
    py1 = py0 + (fy1 - fy0)
    pz1 = pz0 + (fz1 - fz0)

    # Paste
    output[
        fx0:fx1,
        fy0:fy1,
        fz0:fz1
    ] = tumor_patch[px0:px1, py0:py1, pz0:pz1]
    

    return output